# COMPARATIVA con y sin OCCUPATION

In [17]:
import joblib

# Modelos calibrados
modelo_con_occupation = joblib.load("random_forest_model.joblib")
modelo_sin_occupation = joblib.load("rf_sin_ocupacion.joblib")

# Codificadores y escaladores
encoder_con = joblib.load("label_encoder.joblib")  # codificador común a ambos modelos
encoder_sin = joblib.load("encoder_sin_occupation.joblib")

scaler_con = joblib.load("scaler.joblib")  # scaler entrenado con Occupation
scaler_sin = joblib.load("scaler_sin_occupation.pkl")

# Columnas usadas en el entrenamiento
columnas_con = joblib.load("columnas_entrenamiento.joblib")  # con Occupation
columnas_sin = joblib.load("columnas_entrenamiento_sin_occupation.joblib")  # sin Occupation


In [48]:
pacientes = {
    "A – Apnea": {
        'Age': 57,
        'Gender': 'Male',
        'Sleep Duration': 6.0,
        'Quality of Sleep': 4,
        'Physical Activity Level': 30,
        'Stress Level': 2,
        'Heart Rate': 61,
        'Daily Steps': 500,
        'BMI Category': 'Overweight',
        'Systolic BP': 145,
        'Diastolic BP': 95,
        'Occupation': 'Salesperson'
    },
    "B – Sano": {
        'Age': 31,
        'Gender': 'Female',
        'Sleep Duration': 8.5,
        'Quality of Sleep': 9,
        'Physical Activity Level': 80,
        'Stress Level': 1,
        'Heart Rate': 58,
        'Daily Steps': 10000,
        'BMI Category': 'Normal',
        'Systolic BP': 110,
        'Diastolic BP': 80,
        'Occupation': 'Teacher'
    },
    "C – Insomnio": {
        'Age': 55,
        'Gender': 'Male',
        'Sleep Duration': 7.0,
        'Quality of Sleep': 2,
        'Physical Activity Level': 30,
        'Stress Level': 8,
        'Heart Rate': 68,
        'Daily Steps': 6000,
        'BMI Category': 'Normal',
        'Systolic BP': 125,
        'Diastolic BP': 85,
        'Occupation': 'Salesperson'
    }
}

In [49]:
def procesar_paciente_con_occupation(paciente_dict, encoder, scaler, columnas_esperadas):
    df = pd.DataFrame([paciente_dict])

    # Codificar BMI
    bmi_mapping = {'Normal': 1, 'Overweight': 2, 'Obese': 3}
    df['BMI Category'] = df['BMI Category'].map(bmi_mapping)

    # One-hot encoding de variables categóricas
    df = pd.get_dummies(df, columns=['Gender', 'Occupation'])

    # Asegurar que todas las columnas esperadas por el scaler estén presentes (antes de escalar)
    for col in scaler.feature_names_in_:
        if col not in df.columns:
            df[col] = 0

    # Reordenar y aplicar escalado a las columnas vistas por el scaler
    df_scaled = df.copy()
    df_scaled[scaler.feature_names_in_] = scaler.transform(df[scaler.feature_names_in_])

    # Asegurar que todas las columnas esperadas por el modelo estén presentes
    for col in columnas_esperadas:
        if col not in df_scaled.columns:
            df_scaled[col] = 0

    # Reordenar columnas finales
    return df_scaled[columnas_esperadas]


In [50]:
def procesar_paciente_sin_occupation(paciente_dict, scaler, columnas_esperadas):
    df = pd.DataFrame([paciente_dict])

    # Codificar BMI
    bmi_mapping = {'Normal': 1, 'Overweight': 2, 'Obese': 3}
    df['BMI Category'] = df['BMI Category'].map(bmi_mapping)

    # One-hot solo de Gender
    df = pd.get_dummies(df, columns=['Gender'])

    # Asegurar que todas las columnas esperadas por el scaler estén presentes
    for col in scaler.feature_names_in_:
        if col not in df.columns:
            df[col] = 0

    # Escalar
    df_scaled = df.copy()
    df_scaled[scaler.feature_names_in_] = scaler.transform(df[scaler.feature_names_in_])

    # Asegurar columnas del modelo
    for col in columnas_esperadas:
        if col not in df_scaled.columns:
            df_scaled[col] = 0

    return df_scaled[columnas_esperadas]


In [51]:
resultados = []

for nombre, datos in pacientes.items():
    entrada_con = procesar_paciente_con_occupation(datos, encoder_con, scaler_con, columnas_con)
    entrada_sin = procesar_paciente_sin_occupation(datos, scaler_sin, columnas_sin)

    # Predicciones
    pred_con = modelo_con_occupation.predict(entrada_con)[0]
    prob_con = modelo_con_occupation.predict_proba(entrada_con).max()

    pred_sin_id = modelo_sin_occupation.predict(entrada_sin)[0]
    pred_sin = encoder_sin.inverse_transform([pred_sin_id])[0]
    prob_sin = modelo_sin_occupation.predict_proba(entrada_sin).max()

    resultados.append({
        "Paciente": nombre,
        "Predicción con Occupation": f"{pred_con} ({prob_con:.2f})",
        "Predicción sin Occupation": f"{pred_sin} ({prob_sin:.2f})"
    })

df_resultados = pd.DataFrame(resultados)
display(df_resultados)

C:\Users\Marina\anaconda3\lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
C:\Users\Marina\anaconda3\lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
C:\Users\Marina\anaconda3\lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
C:\Users\Marina\anaconda3\lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
C:\Users\Marina\anaconda3\lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
C:\Users\Marina\anaconda3\lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but RandomForestClass

,Paciente,Predicción con Occupation,Predicción sin Occupation
0,A – Apnea,Sleep Apnea (0.70),Insomnia (0.59)
1,B – Sano,Sleep Apnea (0.59),None (0.98)
2,C – Insomnio,Sleep Apnea (0.54),None (0.79)


In [7]:
import joblib

columnas = [
    'Age', 'Sleep Duration', 'Quality of Sleep', 'Physical Activity Level',
    'Stress Level', 'BMI Category', 'Heart Rate', 'Daily Steps',
    'Gender_Male', 'Occupation_Doctor', 'Occupation_Engineer',
    'Occupation_Lawyer', 'Occupation_Manager', 'Occupation_Nurse',
    'Occupation_Sales Representative', 'Occupation_Salesperson',
    'Occupation_Scientist', 'Occupation_Software Engineer',
    'Occupation_Teacher', 'Systolic BP', 'Diastolic BP'
]

joblib.dump(columnas, "columnas_entrenamiento.joblib")

['columnas_entrenamiento.joblib']

In [167]:
import joblib
import pandas as pd

# Carga el modelo entrenado
modelo = joblib.load("random_forest_model.joblib")

# Extrae y guarda las columnas exactamente como el modelo espera
columnas_correctas = list(modelo.feature_names_in_)
print("Orden correcto:", columnas_correctas)

# Guárdalas para la app
joblib.dump(columnas_correctas, "columnas_entrenamiento.joblib")


Orden correcto: ['Age', 'Sleep Duration', 'Quality of Sleep', 'Physical Activity Level', 'Stress Level', 'BMI Category', 'Heart Rate', 'Daily Steps', 'Gender_Male', 'Occupation_Doctor', 'Occupation_Engineer', 'Occupation_Lawyer', 'Occupation_Manager', 'Occupation_Nurse', 'Occupation_Sales Representative', 'Occupation_Salesperson', 'Occupation_Scientist', 'Occupation_Software Engineer', 'Occupation_Teacher', 'Systolic BP', 'Diastolic BP']


['columnas_entrenamiento.joblib']